In [1]:
# 路径引导：无论从项目根还是 notebooks/ 目录启动内核，都能找到 src 包
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('项目根：', ROOT)

项目根： D:\agent项目集\端到端电商智能决策平台


# 04 模拟 A/B 实验分析

**诚实边界**：Olist 是观测数据，不含随机实验，任何“效果”都不能直接作因果解释。
本 notebook 展示的是独立的**模拟** A/B 模块（`src/experiment/`，seed=2026）：
在合成用户上随机分组，完整走一遍实验分析流程 —— SRM 检查、比例检验、效应量、
护栏指标、分层分析、功效分析。流程本身可直接迁移到真实实验数据。

详细报告见 `reports/experiment_report.md`。

In [2]:
import json
import pandas as pd
from src import config

s = json.loads((config.MODEL_DIR / 'experiment_summary.json').read_text(encoding='utf-8'))
print('样本量：{:,}（对照 {:,} / 实验 {:,}）'.format(s['n_users'], s['n_control'], s['n_treatment']))
print('SRM 检验 p 值：%.4f（> 0.05 说明随机分组正常）' % s['srm_p_value'])

样本量：80,000（对照 39,851 / 实验 40,149）
SRM 检验 p 值：0.2921（> 0.05 说明随机分组正常）


## 1. 主指标结果（转化率与 ARPU）

In [3]:
primary = pd.Series({
    '对照组转化率': s['control_rate'],
    '实验组转化率': s['treatment_rate'],
    '绝对差': s['conversion_diff'],
    '转化率检验 p 值': s['conversion_p_value'],
    'ARPU 差': s['arpu_diff'],
    'ARPU 检验 p 值': s['arpu_p_value'],
    '事后功效': s['achieved_power'],
}).round(4)
primary

对照组转化率         0.0331
实验组转化率         0.0360
绝对差            0.0029
转化率检验 p 值      0.0238
ARPU 差         1.2670
ARPU 检验 p 值    0.0010
事后功效           0.6197
dtype: float64

## 2. 护栏指标（确保实验没有误伤其他体验）

In [4]:
guardrails = pd.DataFrame(s['guardrails'])
guardrails.round(4)

,metric,control_rate,treatment_rate,diff,ci_low,ci_high,p_value,relative_lift
0,退款率,0.0249,0.0250,0.0001,-0.0020,0.0023,0.9178,0.0046
1,投诉率,0.0111,0.0121,0.0010,-0.0005,0.0025,0.1807,0.0914


## 3. 分层分析（地区异质性）

In [5]:
regions = pd.DataFrame(s['regions'])
regions.round(4)

,group,n_control,n_treatment,control_rate,treatment_rate,diff,relative_lift,ci_low,ci_high,p_value,p_adjusted,significant_raw,significant_bh
0,东南部,18155,18058,0.0356,0.0406,0.0051,0.1423,0.0011,0.0090,0.0118,0.0592,True,False
1,南部,7808,8076,0.0353,0.0380,0.0027,0.0754,-0.0032,0.0085,0.3718,0.8892,False,False
2,东北部,5924,6059,0.0258,0.0276,0.0017,0.0672,-0.0040,0.0075,0.5558,0.8892,False,False
3,中西部,4761,4776,0.0349,0.0345,-0.0003,-0.0091,-0.0077,0.0070,0.9322,0.9322,False,False
4,北部,3203,3180,0.0240,0.0226,-0.0014,-0.0582,-0.0088,0.0060,0.7114,0.8892,False,False


## 4. 商业价值估算

In [6]:
pd.Series(s['business_value'])

arpu_lift                                1.266989
conversion_lift                          0.002918
incremental_revenue_per_user             1.266989
incremental_revenue_per_100k        126698.947065
incremental_conversions_per_100k       291.773611
dtype: float64

## 5. 功效分析演示：给定基线转化率与 MDE，需要多少样本？

In [7]:
from src.statistics import inference

scenarios = []
for mde in (0.005, 0.01, 0.02, 0.05):
    r = inference.required_sample_size_proportion(baseline_rate=0.10, expected_rate=0.10 + mde)
    scenarios.append({'基线转化率': 0.10, 'MDE(绝对)': mde,
                      '每组所需样本量': r['n_per_group'] if isinstance(r, dict) else r})
pd.DataFrame(scenarios)

,基线转化率,MDE(绝对),每组所需样本量
0,0.1,0.005,57756.0
1,0.1,0.010,14745.0
2,0.1,0.020,3835.0
3,0.1,0.050,681.0


> 结论：MDE 越小，所需样本量呈平方级增长。实验设计前必须先明确
> “业务上值得检测的最小效应”，否则实验要么浪费流量要么功效不足。